# 1. Libraries & Sample Data
The first step is to load our Python Libraries and download the sample data. The dataset represents Apple stock price (1d bars) for the year 2010

In [ ]:
# Load Python Libraries
import math
import keras
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from tqdm.notebook import tqdm
from collections import deque
from IPython.display import display, HTML
from sklearn.preprocessing import StandardScaler

# for dataframe display
pd.set_option("display.max_rows", None)


def display_df(df):
    # Puts the scrollbar next to the DataFrame
    display(
        HTML(
            "<div style='height: 200px; overflow: auto; width: fit-content'>"
            + df.to_html()
            + "</div>"
        )
    )


# for reproducibility of training rounds
keras.utils.set_random_seed(42)

In [ ]:
# Download Sample Data
data = pd.read_csv("GOOG_2009-2010_6m_RAW_1d.csv")

In [ ]:
# check the data
data.head()

In [ ]:
data.info()

# 2. Exploratory Data Analysis
Next, we want to analyze our data. Display the data as a dataframe, and plot some relevant data so you can get an idea of what our dataset looks like.

In [ ]:
# Display as Dataframe
display_df(data)

In [ ]:
# Convert Date column and set as index
data["Date"] = pd.to_datetime(data["Date"])
data = data.set_index("Date").sort_index()

In [ ]:
data.info()
data.head()

In [ ]:
# Quick EDA plot
data["Close"].plot(figsize=(12, 4), title="GOOG Close Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

data.info()

# 3. Data Cleaning
Next, we need to clean our data for training our model. This requires removal of NaN values.

In [ ]:
# Check for null values
data.isnull().sum()

In [ ]:
# forward fill missing values
data = data.replace(0, np.nan).ffill()

In [ ]:
# Check for null values
data.isnull().sum()

In [ ]:
# Plot the cleaned Close Data
data["Close"].plot(figsize=(12, 4), title="GOOG Close Price after ffill")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

data.info()

# 4. Feature Selection
Now that we have cleaned our stock data, we need to select which features to train our model on. For this project, we will be training with Close data and 20-day Bollinger Bands of Close.

In [ ]:
# Calculate 20-day bollinger bands
window_size = 20  # days
num_std = 2

# Calculate Bollinger Bands
data["BB_Middle"] = data["Close"].rolling(window=window_size).mean()  # 20-day SMA
data["BB_Std"] = data["Close"].rolling(window=window_size).std()  # 20-day Std Dev

data["BB_Upper"] = data["BB_Middle"] + (num_std * data["BB_Std"])  # Upper band
data["BB_Lower"] = data["BB_Middle"] - (num_std * data["BB_Std"])  # Lower band

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
# Remove rows with NaN bollinger bands
data.dropna(inplace=True)
data.isnull().sum()

In [ ]:
# Define new dataframe with only the training features (Close, Upper BB, Lower BB)
dataset = data[["Close", "BB_Upper", "BB_Lower"]].copy()

In [ ]:
print(dataset.head(), "\n")
print(dataset.shape, "\n")
print(dataset.isnull().sum())

# 5. Normalization
Now that we have cleaned our data, created our indicators of interest, and selected our features, we must normalize our data. For this project, we use the sklearn StandardScaler, which centers the data and normalizes to unit variance. We will not be using a rolling scaler for this project, due to the complexity of back-translating to true proce and indicator values - you can try this yourself once you have completed the project. 

In [ ]:
# Display & Plot Un-normalized Dataset
dataset["Close"].plot(figsize=(12, 4), title="GOOG Close Price before normalization")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

data.info()

In [ ]:
# Normalize Dataset with StandardScaler
normlist = []
normed_dataset = pd.DataFrame(index=dataset.index)
for col in dataset.columns:
    normalizer = StandardScaler()
    # fit normalizer to column data
    # transform column data with the fitted normalizer, and place the transformed data column in out normed_dataset df
    col_scaled = normalizer.fit_transform(dataset[[col]])
    normed_dataset[col] = col_scaled
    # append the fitted normalizer to normlist for use later
    normlist.append(normalizer)

print(normed_dataset.head())

In [ ]:
# Scaled data should have mean ≈ 0 and std ≈ 1
print("Normed dataset mean:\n", normed_dataset.mean().round(4), "\n")
print("Normed dataset std:\n", normed_dataset.std().round(4), "\n")

In [ ]:
# Display & Plot Normalized Dataset
normed_dataset[["Close", "BB_Upper", "BB_Lower"]].plot(
    figsize=(12, 4), title="GOOG Normalized Features"
)
plt.xlabel("Date")
plt.ylabel("Normalized Price")
plt.legend()
plt.show()

# 6. Train / Test Split
Now that our data cleaned, features are selected, and the dataset is normalized, we are ready to feed the data into our model. With this in mind, we split the data ito train and test data (50/50 split)

In [ ]:
# split dataset df into train (50%) and test (50%) datasets
training_rows = int(len(normed_dataset) * 0.5)

train_df = normed_dataset.iloc[:training_rows]
test_df = normed_dataset.iloc[training_rows + 1 :]


# Verify the split
print(f"Total rows:     {len(normed_dataset)}")
print(f"Training rows:  {len(train_df)}")
print(f"Test rows:      {len(test_df)}")
print(f"Train period:   {train_df.index[0]} → {train_df.index[-1]}")
print(f"Test period:    {test_df.index[0]} → {test_df.index[-1]}")

In [ ]:
# display train and test dfs (ensure no overlap)
print(train_df)

In [ ]:
print(test_df)

In [ ]:
# convert train and test dfs to np arrays with dtype=float
X_train = train_df.values.astype(float)
X_test = test_df.values.astype(float)
# print the shape of X_train to remind yourself how many examples and features are in the dataset
X_train.shape
# track index to remember which feature is which
idx_close = train_df.columns.get_loc("Close")  # → 0
idx_bb_upper = train_df.columns.get_loc("BB_Upper")  # → 1
idx_bb_lower = train_df.columns.get_loc("BB_Lower")  # → 2

In [ ]:
print(train_df.columns.tolist())  # ['Close', 'BB_Upper', 'BB_Lower']

# Double-check by printing first row of each feature
print(f"Close    → column {idx_close}:    {X_train[0, idx_close]:.4f}")
print(f"BB_Upper → column {idx_bb_upper}: \t{X_train[0, idx_bb_upper]:.4f}")
print(f"BB_Lower → column {idx_bb_lower}: \t{X_train[0, idx_bb_lower]:.4f}")

# 7. Define the Agent
Now that our data is ready to use, we can define the Reinforcement Learning Agent.

### Define the DQN Model
The first step in defining our agent is the Deep Q-Network model definition. For this project, we are creating a model sequential model with four layers. The first three layers have output shape of 64, 32, and 8, respectively, and a RELU activation. The output layer has an output shape of the size of our action space (buy, sell, hold), and a linear activation. Our Loss function is Mean Squared Error, and our optimizer is Adam with a learning rate of 0.001. Use Keras to build this model.

In [ ]:
@keras.saving.register_keras_serializable()
class DQN(keras.Model):
    def __init__(self, state_size, action_size):
        super().__init__()

        self.model = keras.models.Sequential(
            [
                keras.layers.Dense(units=64, input_dim=state_size, activation="relu"),
                keras.layers.Dense(units=32, activation="relu"),
                keras.layers.Dense(units=8, activation="relu"),
                keras.layers.Dense(units=action_size, activation="linear"),
            ]
        )

        self.model.compile(
            loss="mse", optimizer=keras.optimizers.Adam(learning_rate=0.001)
        )

    def call(self, x):
        return self.model(x)

### Define Agent Class
Now that we have defined our underlying DQN Model, we must define out Reinforcement Learning Agent. The agent initialization is provided for you, you must define an act function, and an expereince replay function. As a reminder, the act function defines how our model will act (buy, hold, or sell) given a certain state. The Experience Replay function tackles catastrophic forgetting in our training process, by maintaining a memory buffer to allow training on independent / randomized minibatches of previous states. 

In [ ]:
class Agent:
    def __init__(self, window_size, num_features, test_mode=False, model_name=""):
        self.window_size = window_size  # How many days of historical data do we want to include in our state representation?
        self.num_features = num_features  # How many training features do we have?
        self.state_size = (
            window_size * num_features
        )  # State size includes number of training features per day, and number of lookback days
        self.action_size = 3  # 0=hold, 1=buy, 2=sell
        self.memory = deque(
            maxlen=1000
        )  # Bound memory size: once the memory reaches 1000 units, the lefthand values are discarded as righthand values are added
        self.inventory = []  # Inventory to hold trades
        self.model_name = model_name  # filename for saved model checkpoint loading
        self.test_mode = (
            test_mode  # flag for testing (allows model load from checkpoint model_name)
        )

        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

        self.model = keras.models.load_model(model_name) if test_mode else self._model()

    # Deep Q Learning (DQL) model
    def _model(self):
        model = DQN(self.state_size, self.action_size).model
        return model

    # DQL Predict (with input reshaping)
    #   Input = State
    #   Output = Q-Table of action Q-Values
    def get_q_values_for_state(self, state):
        return self.model.predict(state.flatten().reshape(1, self.state_size))

    # DQL Fit (with input reshaping)
    #   Input = State, Target Q-Table
    #   Output = MSE Loss between Target Q-Table and Actual Q-Table for State
    def fit_model(self, input_state, target_output):
        return self.model.fit(
            input_state.flatten().reshape(1, self.state_size),
            target_output,
            epochs=1,
            verbose=0,
        )

    # Agent Action Selector
    #   Input = State
    #   Policy = epsilon-greedy (to minimize possibility of overfitting)
    #   Initially high epsilon = more random, epsilon decay = less random later
    #   Output = Action (0, 1, or 2)
    def act(self, state):
        # Choose any action at random (Probability = epsilon for training mode, 0% for testing mode)
        if not self.test_mode and random.random() <= self.epsilon:
            return random.randrange(self.action_size)
        # Choose the action which has the highest Q-value (Probability = 1-epsilon for training mode, 100% for testing mode)
        # **use model to select action here - i.e. use model to assign q-values to all actions in action space (buy, sell, hold)**
        # **return the action that has the highest value from the q-value function.**
        options = self.get_q_values_for_state(state)
        return np.argmax(options[0])

    # Experience Replay (Learning Function)
    #   Input = Batch of (state, action, next_state) tuples
    #   Optimal Q Selection Policy = Bellman equation
    #   Important Notes = Model fitting step is in this function (fit_model)
    #                     Epsilon decay step is in this function
    #   Output = Model loss from fitting step
    def exp_replay(self, batch_size, losses):
        mini_batch = []
        l = len(self.memory)
        # FIX: use range(l-batch_size, l) to collect exactly batch_size samples
        for i in range(l - batch_size, l):
            mini_batch.append(self.memory[i])

        for state, action, reward, next_state, done in mini_batch:
            # reminders:
            #   - state is a vector containing close & MA values for the current time step
            #   - action is an integer representing the action taken by the act function at the current time step: buy, hold, or sell
            #   - reward represents the profit of a given action - it is either 0 (for buy, hold, and sells which loose money) or the profit in dollars (for a profitable sell)
            #   - next_state is a vector containing close & MA values for the next time step
            #   - done is a boolean flag representing whether or not we are in the last iteration of a training episode (i.e. True when next_state does not exist.)
            if done:
                # special condition for last training epoch in batch (no next_state)
                optimal_q_for_action = reward
            else:
                # target Q-value is updated using the Bellman equation: reward + gamma * max(predicted Q-value of next state)
                optimal_q_for_action = reward + self.gamma * np.max(
                    self.get_q_values_for_state(next_state)
                )

            # Get the predicted Q-values of the current state
            target_q_table = self.get_q_values_for_state(state)
            # Update the output Q table - replace the predicted Q value for action with the target Q value for action
            target_q_table[0][action] = optimal_q_for_action
            # Fit the model where state is X and target_q_table is Y
            history = self.fit_model(state, target_q_table)
            losses += history.history["loss"]

        # define epsilon decay (for the act function)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        return losses

# 8. Train the Agent
Now that our data is ready and our agent is defined, we are ready to train the agent. 

### Helper Functions
Before we define the training loop, we will write some helper functions: one for printing price data, one to define the sigmoid funtion, one to grab the state representation,  one to plot the trading output of our trained model, and one to plot the training loss. The printing, sigmoid, and plotting functions are defined for you. You must define the function which gets the state representation.

In [ ]:
# Format price string
def format_price(n):
    return ("-$" if n < 0 else "$") + "{0:.2f}".format(abs(n))


def sigmoid(x):
    return 1 / (1 + math.exp(-x))


# Plot behavior of trade output
def plot_behavior(
    data_input,
    bb_upper_data,
    bb_lower_data,
    states_buy,
    states_sell,
    profit,
    train=True,
):
    fig = plt.figure(figsize=(15, 5))
    plt.plot(data_input, color="k", lw=2.0, label="Close Price")
    plt.plot(bb_upper_data, color="b", lw=2.0, label="Bollinger Bands")
    plt.plot(bb_lower_data, color="b", lw=2.0)
    plt.plot(
        data_input,
        "^",
        markersize=10,
        color="r",
        label="Buying signal",
        markevery=states_buy,
    )
    plt.plot(
        data_input,
        "v",
        markersize=10,
        color="g",
        label="Selling signal",
        markevery=states_sell,
    )
    plt.title("Total gains: %f" % (profit))
    plt.legend()
    if train:
        plt.xticks(
            range(0, len(train_df.index.values), int(len(train_df.index.values) / 15)),
            train_df.index.values[0 :: int(len(train_df.index.values) / 15)],
            rotation=45,
            fontsize="small",
        )
    else:
        plt.xticks(
            range(0, len(test_df.index.values), int(len(test_df.index.values) / 15)),
            test_df.index.values[0 :: int(len(test_df.index.values) / 15)],
            rotation=45,
            fontsize="small",
        )
    plt.show()


# Plot training loss
def plot_losses(losses, title):
    plt.plot(losses)
    plt.title(title)
    plt.ylabel("MSE Loss Value")
    plt.xlabel("batch")
    plt.show()


# returns an an n-day state representation ending at time t
def get_state(data, t, n):
    # data is the dataset of interest which holds the state values (i.e. Close , BB Upper, BB Lower)
    # t is the current time step
    # n is the size of the training window
    # FIX: use d = t - n + 1 so that block includes the current timestep t
    d = t - n + 1
    if d >= 0:
        block = data[d : t + 1]  # n elements from t-(n-1) to t (inclusive)
    else:
        # pad with copies of the first row when there is not enough history
        block = np.vstack([np.tile(data[0], (-d, 1)), data[: t + 1]])

    # apply sigmoid to each consecutive difference in the window
    res = []
    for i in range(n - 1):
        feature_res = []
        for feature in range(data.shape[1]):
            feature_res.append(sigmoid(block[i + 1, feature] - block[i, feature]))
        res.append(feature_res)
    return np.array([res])

### Training Loop

In [ ]:
# display the shape of your training data in order to remind yourself how may features and examples there are in your training set
X_train.shape

In [ ]:
keras.utils.disable_interactive_logging()  # disable built-in keras loading bars - they make the output difficult to read and monitor

window_size = 1

agent = Agent(window_size, num_features=X_train.shape[1])

In [ ]:
normed_dataset.head()

In [ ]:
keras.config.disable_traceback_filtering()  # disable built-in keras loading bars - they make the output difficult to read and monitor


l = X_train[:, 0].shape[0] - 1

# batch size defines how often to run the exp_replay method
batch_size = 32

# An episode represents a complete pass over the data.
episode_count = 2

normalizer_close = normlist[idx_close]
normalizer_bb_upper = normlist[idx_bb_upper]
normalizer_bb_lower = normlist[idx_bb_lower]

X_train_true_price = normalizer_close.inverse_transform(
    X_train[:, idx_close].reshape(-1, 1)
).flatten()
X_train_true_bb_upper = normalizer_bb_upper.inverse_transform(
    X_train[:, idx_bb_upper].reshape(-1, 1)
).flatten()
X_train_true_bb_lower = normalizer_bb_lower.inverse_transform(
    X_train[:, idx_bb_lower].reshape(-1, 1)
).flatten()

batch_losses = []
num_batches_trained = 0

for e in range(episode_count + 1):
    state = get_state(X_train, 0, window_size + 1)
    # initialize variables
    total_profit = 0
    total_winners = 0
    total_losers = 0
    agent.inventory = []
    states_sell = []
    states_buy = []
    for t in tqdm(range(l), desc=f"Running episode {e}/{episode_count}"):
        # get the action
        action = agent.act(state)
        # get the next state
        next_state = get_state(X_train, t + 1, window_size + 1)

        # initialize reward for the current time step
        reward = 0

        if action == 1:  # buy
            # inverse transform to get true buy price in dollars
            # append the buy price to the inventory
            # append the time step to states_buy
            # print the action and price of the action
            buy_price = X_train_true_price[t]
            agent.inventory.append(buy_price)
            states_buy.append(t)
            print(f"Buy: {format_price(buy_price)}")

        elif action == 2 and len(agent.inventory) > 0:  # sell
            # get the bought price of the stock you are selling (i.e. the stock at the beginning of the inventory)
            bought_price = agent.inventory.pop(0)
            # inverse transform to get true sell price in dollars
            # define reward as max of profit (close price at time of sell - close price at time of buy) and 0
            # add current profit to total profit
            sell_price = X_train_true_price[t]
            trade_profit = sell_price - bought_price
            reward = max(trade_profit, 0)
            total_profit += trade_profit
            if trade_profit >= 0:
                # add current profit to total winners
                total_winners += trade_profit
            else:
                # add current profit to total losers
                total_losers += trade_profit
            # append the time step to states_sell
            states_sell.append(t)
            # print the action, price of the action, and profit of the action
            print(
                f"Sell: {format_price(sell_price)} | Profit: {format_price(trade_profit)}"
            )

        # flag for final training iteration
        done = True if t == l - 1 else False

        # append the details of the state action etc in the memory, to be used by the exp_replay function
        agent.memory.append((state, action, reward, next_state, done))
        state = next_state

        # print total profit and plot behavior of the current episode when the episode is finished
        if done:
            print("--------------------------------")
            print(f"Episode {e}")
            print(f"Total Profit: {format_price(total_profit)}")
            print(f"Total Winners: {format_price(total_winners)}")
            print(f"Total Losers: {format_price(total_losers)}")
            print(
                f"Max Loss: {max(batch_losses[num_batches_trained:len(batch_losses)])}"
            )
            print(
                f"Total Loss: {sum(batch_losses[num_batches_trained:len(batch_losses)])}"
            )
            print("--------------------------------")
            plot_behavior(
                X_train_true_price,
                X_train_true_bb_upper,
                X_train_true_bb_lower,
                states_buy,
                states_sell,
                total_profit,
            )
            plot_losses(
                batch_losses[num_batches_trained : len(batch_losses)],
                f"Episode {e} DQN model loss",
            )
            num_batches_trained = len(batch_losses)

        # when the size of the memory is greater than the batch size, run the exp_replay function on the batch to fit the model and get losses for the batch
        # then sum the losses for the batch and append them to the batch_losses list
        if len(agent.memory) > batch_size:
            losses = agent.exp_replay(batch_size, [])
            batch_losses.append(sum(losses))

    if e % 2 == 0:
        # save the model every 2 episodes (in case of crash or better training iteration in the middle of training process)
        agent.model.save(f"model_ep{e}.keras")

# always save the final model so the test section can load it
agent.model.save(f"model_ep{episode_count}.keras")

### Plot Training Loss

In [ ]:
# use the plot_losses function to plot all batch_losses for the entire training round
plot_losses(batch_losses, "Training Loss - All Episodes")

# 9. Test the trained agent 
Finally, we get to test our trained model to see how well it performs in our test set. Using the training loop above, define a method to run our trained model on our X_test dataset. 

### Define Parameters
Some test parameters are defined for you below. Fill out the missing data. If you need a hint, look up at the training loop. 

In [ ]:
l_test = len(X_test) - 1
state = get_state(X_test, 0, window_size + 1)
total_profit = 0
total_winners = 0
total_losers = 0
done = False
states_sell_test = []
states_buy_test = []

# Get the trained model
agent = Agent(
    window_size,
    num_features=X_test.shape[1],
    test_mode=True,
    model_name=f"model_ep{episode_count}.keras",
)
agent.inventory = []

X_test_true_price = normalizer_close.inverse_transform(
    X_test[:, idx_close].reshape(-1, 1)
).flatten()
X_test_true_bb_upper = normalizer_bb_upper.inverse_transform(
    X_test[:, idx_bb_upper].reshape(-1, 1)
).flatten()
X_test_true_bb_lower = normalizer_bb_lower.inverse_transform(
    X_test[:, idx_bb_lower].reshape(-1, 1)
).flatten()

### Run the Test
Run the test data through the trained model. Look at the training loop for a hint.

In [ ]:
for t in range(l_test):
    action = agent.act(state)
    next_state = get_state(X_test, t + 1, window_size + 1)
    reward = 0

    if action == 1:  # buy
        # inverse transform to get true buy price in dollars
        # append buy price to inventory
        # append time step to states_buy_test
        buy_price = X_test_true_price[t]
        agent.inventory.append(buy_price)
        states_buy_test.append(t)
        print(f"Buy: {format_price(buy_price)}")

    elif action == 2 and len(agent.inventory) > 0:  # sell
        # get bought price from beginning of inventory
        # inverse transform to get true sell price in dollars
        # reward is max of profit (close price at time of sell - close price at time of buy)
        # update total_test_profit
        # append time step to states_sell_test
        bought_price = agent.inventory.pop(0)
        sell_price = X_test_true_price[t]
        trade_profit = sell_price - bought_price
        reward = max(trade_profit, 0)
        total_profit += trade_profit
        if trade_profit >= 0:
            # add current profit to total winners
            total_winners += trade_profit
        else:
            # add current profit to total losers
            total_losers += trade_profit
        states_sell_test.append(t)

        print(
            f"Sell: {format_price(sell_price)} | Profit: {format_price(sell_price - bought_price)}"
        )

    if t == l_test - 1:
        done = True

    # append to memory so we can re-train on 'live' (test) data later
    agent.memory.append((state, action, reward, next_state, done))
    state = next_state

    if done:
        print("------------------------------------------")
        print(f"Total Profit: {format_price(total_profit)}")
        print("------------------------------------------")

plot_behavior(
    X_test_true_price,
    X_test_true_bb_upper,
    X_test_true_bb_lower,
    states_buy_test,
    states_sell_test,
    total_profit,
    train=False,
)